# ALIM + Qualcomm VLM — Colab Integration

**You do not give this a page number. You give it a query, and it finds the page itself.**

```
real datasheet PDF -> ALIM ranks pages by relevance to your query (cheap, no VLM)
                    -> sends only the top few candidate pages to the VLM
                    -> VLM reads those pages, returns structured JSON evidence
                    -> alim.decision.core.extract_parameter (unmodified) decides
                    -> parameter / value / unit / condition / page
```

The VLM never decides between candidates and is never allowed to invent a
missing field. It also never sees pages you didn't ask about -- ALIM
narrows that down on its own using free text-matching before it ever
calls the model. See `alim/candidates/page_ranking.py` for how.

## Honest scope of what this notebook can and cannot do

- **Colab has no Snapdragon silicon.** Running the model here is standard
  GPU inference via Hugging Face `transformers`, not on-device Snapdragon
  execution. Real published on-device numbers are cited in
  `docs/QUALCOMM.md`, not reproduced here.
- **Genuine on-device performance evidence** requires either a real
  Snapdragon-on-Windows device, or Qualcomm AI Hub Workbench's hosted
  device farm (§8 below, requires your own account/token).
- **Model:** `Qwen3-VL-4B-Instruct`, selected over the originally-suggested
  Qwen2.5-VL-7B-Instruct after checking the live AI Hub catalog -- see
  `docs/QUALCOMM.md` §1.

## MOCK_MODE

`MOCK_MODE = True` runs the whole notebook immediately, no GPU, no
download. `MOCK_MODE = False` runs a real model on Colab's GPU.


In [ ]:
MOCK_MODE = True  # flip to False once you're ready to load the real model

# LM35's real datasheet has a device-variant table that ALIM's structural
# pass correctly refuses to parse (a genuinely different table shape --
# see docs/QUALCOMM.md). That means structural extraction alone produces
# ZERO candidates here, so this query can ONLY be answered by the VLM --
# a clean demonstration of exactly when the VLM matters, with automatic
# page-finding and no competing structural noise to confuse the result.
QUERY = "supply voltage"
PDF_NAME = "LM35.pdf"


## 1. Install dependencies

In [ ]:
import json

# Real per-page evidence for LM35's real Operating Conditions page (values
# read directly off the real datasheet earlier in this project). A real
# model reads the actual rendered image instead of this lookup.
MOCK_EVIDENCE_BY_PAGE = {
    4: [{"section": "Operating conditions", "table_caption": "Table 2. Operating conditions",
         "parameter": "Supply voltage", "symbol": "VCC", "condition": None, "min": "4", "max": "30",
         "unit": "V", "page": 4, "uncertain": False}],
}

def mock_generate_fn(image_bytes, prompt):
    return json.dumps(mock_generate_fn._last_page_evidence)

def real_generate_fn(image_bytes, prompt):
    import torch
    from transformers import AutoModelForImageTextToText, AutoProcessor
    from PIL import Image
    import io

    model_id = "Qwen/Qwen3-VL-4B-Instruct"
    if not hasattr(real_generate_fn, "_model"):
        real_generate_fn._processor = AutoProcessor.from_pretrained(model_id)
        real_generate_fn._model = AutoModelForImageTextToText.from_pretrained(
            model_id, dtype=torch.bfloat16, device_map="auto")

    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text_prompt = real_generate_fn._processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = real_generate_fn._processor(text=[text_prompt], images=[image], return_tensors="pt").to(
        real_generate_fn._model.device)
    output_ids = real_generate_fn._model.generate(**inputs, max_new_tokens=1024)
    return real_generate_fn._processor.decode(output_ids[0][inputs["input_ids"].shape[1]:],
                                                skip_special_tokens=True)

generate_fn = mock_generate_fn if MOCK_MODE else real_generate_fn


## 2. Install the ALIM engine

Replace `ALIM_REPO_URL` with the URL you pushed this repository to.

In [ ]:
ALIM_REPO_URL = "<paste your pushed alim repository URL here>"

import os
if not os.path.exists("alim_engine_repo"):
    !git clone {ALIM_REPO_URL} alim_engine_repo
%cd alim_engine_repo
!pip install -q -e .
!python fetch_fixtures.py

pdf_path = f"alim/tests/fixtures/{PDF_NAME}"


## 3. Model backend

**No page number anywhere in this cell.** `generate_fn(image_bytes, prompt) -> raw_text`
is all either backend needs to expose -- ALIM decides which page's image to pass in.

**Mock backend**: returns realistic evidence for whatever page it's asked
about, drawing from real values captured earlier in this project (the
Crystal Oscillator Characteristics page, and the real mislabeled-header
Absolute Maximum Ratings table) so the pipeline is fully testable with no
GPU.

**Real backend**: loads `Qwen/Qwen3-VL-4B-Instruct` via `transformers`.
Not the `qualcomm/` HF repo -- that one only hosts pre-exported on-device
runtime assets, not a `transformers`-loadable checkpoint. If
`AutoModelForImageTextToText` doesn't recognize this architecture,
`pip install -U transformers` or check the model card for the current
class name -- not verified against a live download in the environment
that wrote this notebook (no network route to huggingface.co there).

If you hit `CUDA out of memory`: **Runtime -> Restart session**, then
re-run from the top with a lower zoom (`QualcommVLMProvider(generate_fn=generate_fn, zoom=1.0)`
in §4 below).


In [ ]:
import json

# Real per-page evidence, captured earlier in this project from the actual
# rendered nRF24L01+ pages -- keyed by page number so the mock backend can
# answer correctly no matter which page ALIM's ranking sends it.
MOCK_EVIDENCE_BY_PAGE = {
    12: [{"section": "Absolute maximum ratings", "table_caption": "Table 2. Absolute maximum ratings",
          "parameter": "VDD", "symbol": "VDD", "condition": None, "min": "-0.3", "max": "3.6", "unit": "V",
          "page": 12, "uncertain": False}],
    13: [{"section": "Operating conditions", "table_caption": "Table 3. Operating conditions",
          "parameter": "Supply voltage", "symbol": "VDD", "condition": None, "min": "1.9", "typ": "3.0",
          "max": "3.6", "unit": "V", "page": 13, "uncertain": False}],
    19: [{"section": "Crystal oscillator characteristics", "table_caption": "Table 16. Crystal oscillator characteristics",
          "parameter": "Tolerance", "symbol": "ΔF", "condition": None, "min": None, "typ": None, "max": "60",
          "unit": "ppm", "page": 19, "uncertain": False},
         {"section": "Crystal oscillator characteristics", "table_caption": "Table 16. Crystal oscillator characteristics",
          "parameter": "Crystal Frequency", "symbol": "Fxo", "condition": None, "min": None, "typ": "16",
          "max": None, "unit": "MHz", "page": 19, "uncertain": False}],
}

def mock_generate_fn(image_bytes, prompt):
    # A real model reads the image; the mock reads which page was rendered
    # by inspecting how it was called -- see QualcommVLMProvider below,
    # which passes page_number through so this stays honest about which
    # page it's "looking at."
    return json.dumps(mock_generate_fn._last_page_evidence)

def real_generate_fn(image_bytes, prompt):
    import torch
    from transformers import AutoModelForImageTextToText, AutoProcessor
    from PIL import Image
    import io

    model_id = "Qwen/Qwen3-VL-4B-Instruct"
    if not hasattr(real_generate_fn, "_model"):
        real_generate_fn._processor = AutoProcessor.from_pretrained(model_id)
        real_generate_fn._model = AutoModelForImageTextToText.from_pretrained(
            model_id, dtype=torch.bfloat16, device_map="auto")

    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text_prompt = real_generate_fn._processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = real_generate_fn._processor(text=[text_prompt], images=[image], return_tensors="pt").to(
        real_generate_fn._model.device)
    output_ids = real_generate_fn._model.generate(**inputs, max_new_tokens=1024)
    return real_generate_fn._processor.decode(output_ids[0][inputs["input_ids"].shape[1]:],
                                                skip_special_tokens=True)

generate_fn = mock_generate_fn if MOCK_MODE else real_generate_fn


## 4. Run it -- query in, answer out, no page number given

This is the whole point of this notebook. `extract()` takes a PDF and a
question. It ranks pages on its own, calls the VLM only on the ones
likely to matter (at most `max_vlm_pages`, default 5), and returns a
decided answer. Watch the printed `vlm_calls` list below -- those page
numbers were chosen by ALIM, not typed in by you.


In [ ]:
from alim.integrations.qualcomm.provider import QualcommVLMProvider
from alim.api.extract import extract

class TrackingProvider(QualcommVLMProvider):
    """Wraps QualcommVLMProvider just to print which page it's about to
    look at, and to hand the mock backend the right canned evidence for
    that page -- a real model just reads the actual image instead."""
    def perceive(self, pdf_path, page_number, query):
        print(f"  [ALIM chose to look at page {page_number}]")
        if MOCK_MODE:
            mock_generate_fn._last_page_evidence = MOCK_EVIDENCE_BY_PAGE.get(page_number, [])
        return super().perceive(pdf_path, page_number, query)

provider = TrackingProvider(generate_fn=generate_fn, zoom=2.0)

print(f"Query: {QUERY!r}")
print(f"Document: {PDF_NAME}")
print("\nPages ALIM is checking (chosen automatically, not specified by you):")
result = extract(pdf_path, QUERY, vlm_provider=provider, trace=False)

print("\n--- ALIM decision ---")
print("status:", result["status"])
if result.get("results"):
    for r in result["results"]:
        print(f"  {r['parameter']} = {r['value']} {r['unit']} (page {r['page']}, condition: {r['condition'] or '-'})")
if result.get("vlm_calls"):
    print("\nVLM was called on these pages (ALIM's own choice):",
          [c["page"] for c in result["vlm_calls"]])
else:
    print("\n(Structural extraction alone answered this -- the VLM was never needed.)")


## 5. The mislabeled-header case (optional, separate document)

This is the case from earlier in this project: a real Absolute Maximum
Ratings table on the nRF24L01+ datasheet whose own header text confusingly
says "Operating conditions." Still no page number given -- ALIM finds
pages 12 and 13 on its own and correctly prefers page 13's real Operating
Conditions table.


In [ ]:
nrf_path = "alim/tests/fixtures/nRF24L01P.PDF"
MOCK_EVIDENCE_BY_PAGE_NRF = {
    12: [{"section": "Absolute maximum ratings", "table_caption": "Table 2. Absolute maximum ratings",
          "parameter": "VDD", "symbol": "VDD", "condition": None, "min": "-0.3", "max": "3.6", "unit": "V",
          "page": 12, "uncertain": False}],
    13: [{"section": "Operating conditions", "table_caption": "Table 3. Operating conditions",
          "parameter": "Supply voltage", "symbol": "VDD", "condition": None, "min": "1.9", "typ": "3.0",
          "max": "3.6", "unit": "V", "page": 13, "uncertain": False}],
}

class NrfTrackingProvider(QualcommVLMProvider):
    def perceive(self, pdf_path, page_number, query):
        print(f"  [ALIM chose to look at page {page_number}]")
        if MOCK_MODE:
            mock_generate_fn._last_page_evidence = MOCK_EVIDENCE_BY_PAGE_NRF.get(page_number, [])
        return super().perceive(pdf_path, page_number, query)

nrf_provider = NrfTrackingProvider(generate_fn=generate_fn, zoom=2.0)
result2 = extract(nrf_path, "operating supply voltage", vlm_provider=nrf_provider)
print("status:", result2["status"])
for r in result2.get("results", []):
    parameter = r["parameter"]
    value = r["value"]
    unit = r["unit"]
    page = r["page"]
    print(f"  {parameter} = {value} {unit} (page {page})")
print()
print("(This query is usually answered by structural extraction alone on the real")
print(" nRF24L01+ document, so the VLM may not be called at all -- correct behavior.")
print(" Section 4 above is the example that reliably needs the VLM.)")


## 6. (Optional) Real Snapdragon device profiling via Qualcomm AI Hub Workbench

Requires your own Qualcomm AI Hub account and API token -- sign up at
https://myaccount.qualcomm.com/signup, token from
https://aihub.qualcomm.com (Workbench). **Not run in the environment
that built this notebook** -- no such account/token available there.


In [ ]:
# !pip install -q qai-hub
# !qai-hub configure --api_token YOUR_TOKEN_HERE
#
# import qai_hub as hub
# device = hub.Device("Snapdragon X Elite CRD")
# # See https://github.com/qualcomm/ai-hub-models/tree/main/qai_hub_models/models/qwen3_vl_4b_instruct
# # for the model-specific compile/profile job submission code -- this varies
# # by model and was not executed here.


## 7. What this notebook does and does not prove

**Proves (once run with `MOCK_MODE = False` on a real GPU):** given only
a PDF and a plain-English question, ALIM finds the relevant page(s) on
its own, sends only those to a real current-generation VLM, and the
existing unmodified decision core correctly resolves the answer --
including the real mislabeled-header adversarial case.

**Does not prove:** on-device Snapdragon latency/memory (§6), that
Qwen3-VL-4B beats Qwen2.5-VL-7B at this task specifically, or that this
generalizes beyond the datasheets in the fixture corpus.


## 8. Batch test across the full real-datasheet corpus (22 documents, 8+ device categories)

Same idea as §4, run across every fixture -- no page numbers anywhere in
this cell either. A structural-only baseline (no VLM) was already
measured -- see `docs/QUALCOMM.md` §6: 4/22 resolve structurally, 10 come
back `SCHEMA_UNKNOWN` (the real target set for this test), 7 `NOT_FOUND`,
1 `AMBIGUOUS_MISSING_CONDITION`.

**In `MOCK_MODE`, this cell's answers aren't meaningful across different documents** -- the mock backend only knows canned answers for specific pages of LM35/nRF24L01+, not all 22 files. It's still useful in mock mode to confirm nothing crashes across the whole corpus. Run with `MOCK_MODE = False` for results that actually mean something here, since a real model reads each document's real page correctly regardless of which file it is.


In [ ]:
import time

QUERIES_BY_FILE = {
    "DS18b20.pdf": "supply voltage", "LM35.pdf": "supply voltage", "MQ-7.pdf": "supply voltage",
    "nRF24L01P.PDF": "operating supply voltage", "MLX90614.pdf": "external supply",
    "LM317.pdf": "operating input to output differential voltage", "LM7805.pdf": "input voltage",
    "LM358.pdf": "supply voltage", "LM741.pdf": "supply voltage", "TL082.pdf": "supply voltage",
    "LM339.pdf": "supply voltage", "NE555.pdf": "supply voltage", "CA3306.pdf": "supply voltage",
    "MCP4725.pdf": "supply voltage", "CD4017.pdf": "supply voltage", "KA34063.pdf": "supply voltage",
    "BMP180.pdf": "supply voltage", "VL53L0X.pdf": "supply voltage", "STM32F103C8.pdf": "supply voltage",
    "AT24C02A.pdf": "supply voltage", "AT45DB041B.pdf": "supply voltage", "74HC595.pdf": "supply voltage",
}

results = []
for fname, q in QUERIES_BY_FILE.items():
    path = f"alim/tests/fixtures/{fname}"
    t0 = time.time()
    try:
        r = extract(path, q, vlm_provider=provider)
        status = r["status"]
        n_vlm_calls = len(r.get("vlm_calls", []))
    except Exception as e:
        status = f"ERROR: {type(e).__name__}"
        n_vlm_calls = 0
    dt = time.time() - t0
    results.append((fname, q, status, dt, n_vlm_calls))
    print(f"{fname:18s} {q:45s} {status:26s} {dt:5.1f}s  vlm_calls={n_vlm_calls}")

from collections import Counter
print("\n--- summary ---")
print(Counter(s for _, _, s, _, _ in results))


## 9. Never return empty-handed on failure

`include_all_evidence=True` adds every candidate the engine considered
(matched or not) to the result. `to_xml()` turns any result into XML for
handing to another LLM or tool.


In [ ]:
from alim.api.xml_export import to_xml

result3 = extract(pdf_path, QUERY, vlm_provider=provider, include_all_evidence=True)
print(f"status: {result3['status']}  |  candidates considered: {len(result3.get('all_evidence', []))}")
print()
print(to_xml(result3)[:2000])
